## Celda 1 - contruir la tabla con ceros

In [1]:
import pandas as pd
import numpy as np
import holidays
from pathlib import Path

SALIDA = Path("procesado")
limpios = pd.read_parquet(SALIDA / "viajes_limpios.parquet", columns=["inicio", "origen"])
seleccion = pd.read_csv(SALIDA / "estaciones_seleccionadas.csv")
estaciones = seleccion["origen"].tolist()

# Conteo de salidas por estación y hora
v = limpios.loc[limpios["origen"].isin(estaciones)].copy()
v["hora_inicio"] = v["inicio"].dt.floor("h")
conteo = v.groupby(["origen", "hora_inicio"]).size().rename("salidas")

# Rejilla completa: todas las estaciones x todas las horas -> los faltantes son ceros
horas = pd.date_range("2024-01-01 00:00", "2025-12-31 23:00", freq="h")
indice = pd.MultiIndex.from_product([estaciones, horas], names=["origen", "hora_inicio"])
tabla = conteo.reindex(indice, fill_value=0).reset_index()

# Horario de servicio: 5:00 a 00:59 (se excluyen las horas 1 a 4)
h = tabla["hora_inicio"].dt.hour
fuera = h.between(1, 4)
print(f"Salidas registradas entre 1:00 y 4:59: {tabla.loc[fuera, 'salidas'].sum():,}")
tabla = tabla.loc[~fuera].reset_index(drop=True)

print(f"Filas: {len(tabla):,}  (esperado: 30 estaciones x 731 días x 20 horas = {30*731*20:,})")
print(f"Horas con cero salidas: {(tabla['salidas'] == 0).mean():.1%}")
print(f"Total de salidas en la tabla: {tabla['salidas'].sum():,}  (en viajes filtrados: {len(v):,})")

Salidas registradas entre 1:00 y 4:59: 3
Filas: 438,600  (esperado: 30 estaciones x 731 días x 20 horas = 438,600)
Horas con cero salidas: 12.9%
Total de salidas en la tabla: 2,349,325  (en viajes filtrados: 2,349,328)


## Celda 2 - Variables de calendario, festivos y datos de la estacion

In [2]:
f = tabla["hora_inicio"]
tabla["anio"] = f.dt.year
tabla["mes"] = f.dt.month
tabla["dia_semana"] = f.dt.dayofweek          # 0 = lunes ... 6 = domingo
tabla["hora"] = f.dt.hour
tabla["fin_de_semana"] = (tabla["dia_semana"] >= 5).astype(int)

festivos_mx = holidays.Mexico(years=[2024, 2025])
fechas_festivas = pd.to_datetime(list(festivos_mx.keys()))
tabla["festivo"] = f.dt.normalize().isin(fechas_festivas).astype(int)

print("Días festivos considerados:")
for fecha, nombre in sorted(festivos_mx.items()):
    print(f"  {fecha}  {nombre}")

# Datos fijos de cada estación (desde la nomenclatura)
tabla = tabla.merge(
    seleccion[["origen", "latitude", "longitude", "dpcapacity"]], on="origen", how="left")

print()
print(tabla.head().to_string())

Días festivos considerados:
  2024-01-01  Año Nuevo
  2024-02-05  Día de la Constitución
  2024-03-18  Natalicio de Benito Juárez
  2024-05-01  Día del Trabajo
  2024-09-16  Día de la Independencia
  2024-10-01  Transmisión del Poder Ejecutivo Federal
  2024-11-18  Día de la Revolución
  2024-12-25  Navidad
  2025-01-01  Año Nuevo
  2025-02-03  Día de la Constitución
  2025-03-17  Natalicio de Benito Juárez
  2025-05-01  Día del Trabajo
  2025-09-16  Día de la Independencia
  2025-11-17  Día de la Revolución
  2025-12-25  Navidad

   origen         hora_inicio  salidas  anio  mes  dia_semana  hora  fin_de_semana  festivo  latitude   longitude  dpcapacity
0      51 2024-01-01 00:00:00        0  2024    1           0     0              0        1  20.67414 -103.355535          39
1      51 2024-01-01 05:00:00        0  2024    1           0     5              0        1  20.67414 -103.355535          39
2      51 2024-01-01 06:00:00        0  2024    1           0     6              0   

## Celda 3 - Resumen descriptivo

In [3]:
print(tabla["salidas"].describe().round(2))

print("\nPromedio de salidas por hora del día:")
print(tabla.groupby("hora")["salidas"].mean().round(2).to_string())

print("\nPromedio de salidas por día de la semana (0 = lunes):")
print(tabla.groupby("dia_semana")["salidas"].mean().round(2).to_string())

print("\nPromedio en festivos vs. no festivos:")
print(tabla.groupby("festivo")["salidas"].mean().round(2).to_string())

count    438600.00
mean          5.36
std           5.39
min           0.00
25%           2.00
50%           4.00
75%           7.00
max         205.00
Name: salidas, dtype: float64

Promedio de salidas por hora del día:
hora
0     0.64
5     0.63
6     2.48
7     5.75
8     8.13
9     5.56
10    4.91
11    5.36
12    5.70
13    6.56
14    6.94
15    7.23
16    6.58
17    7.82
18    9.51
19    8.19
20    6.29
21    4.65
22    2.88
23    1.33

Promedio de salidas por día de la semana (0 = lunes):
dia_semana
0    5.58
1    5.94
2    5.97
3    5.92
4    5.72
5    4.31
6    4.05

Promedio en festivos vs. no festivos:
festivo
0    5.41
1    3.02


## Celda 4 - Guardar

In [4]:
tabla.to_parquet(SALIDA / "tabla_modelo.parquet", index=False)
print("Guardado:", SALIDA / "tabla_modelo.parquet", tabla.shape)

Guardado: procesado/tabla_modelo.parquet (438600, 12)
